# Operatori di creazione e distruzione di time-step

In [1]:
from QuantumCompiling import CircuitState, CircuitShape
import scipy.sparse as sp
import numpy as np
import csv

Definizione della classe CircuitOperations, che calcola e tiene in memoria gli operatori |n><m| per spostarsi dal generico time-step m al generico time-step n 

In [2]:
class CircuitOperations:
    def __init__(self,shape):  
        self._shape = shape
        self._hops = np.zeros((len(shape._steps),len(shape._steps)),dtype=object)
        for i in range(len(shape._steps)):
            for j in range(len(shape._steps)):
                self._hops[i,j] = sp.csr_matrix( np.kron(
                    np.conj(shape.state_of_step(shape._steps[j])).reshape(-1,1),
                    shape.state_of_step(shape._steps[i])
                ), dtype='complex'  )

    # moves the single-step-circuit state to "to" single-step-circuit state
    def hop(self,to,circuit):
        if not circuit.get_n_steps() == 1: raise Exception("The circuit is not single-step")
        circuit._state = self._hops[circuit._shape.index_of_state(circuit._state),to].dot(circuit._state) 
    
    def multiply(operator,circuit_state):
        final_state = operator.dot(circuit_state._state)
        return CircuitState(circuit_state._shape,state=final_state)
        
    def get_H_eps(self,ids1,ids2,n_steps):
        id = sp.identity(len(self._hops), format='csr')
        H_eps = sp.csr_matrix((len(self._hops)**n_steps,len(self._hops)**n_steps), dtype='complex')
        for i_eq in range(0,n_steps+1-len(ids1)):
            H_eps_i = sp.identity(1,format='csr') 
            for i in range(0,i_eq):
                H_eps_i = sp.kron(H_eps_i,id)
            for i in range(0,len(ids1)):
                H_eps_i = sp.kron(H_eps_i,self._hops[ids1[i],ids2[i]])
            for i in range(i_eq+len(ids1),n_steps):
                H_eps_i = sp.kron(H_eps_i,id)
            H_eps += H_eps_i
        
        return H_eps + H_eps.getH()
    
    def get_H(self, dir, dmax, n_steps):
        H = sp.csr_matrix((len(self._hops)**n_steps,len(self._hops)**n_steps), dtype='complex')
        for d in range(2,dmax+1):
            filename = f"{self._shape._eq_dir}/eq_d={d}.csv"
            with open(filename, 'r', newline='') as fin:
                reader = csv.reader(fin, delimiter="\t")
                #skips the header
                row = None
                while(row != "equivalences:"):
                    row = next(reader)[0]
                
                for row in reader:
                    ids1 = [int(id) for id in row[:d]]
                    ids2 = [int(id) for id in row[d:]]
                    H += self.get_H_eps(ids1,ids2,n_steps)
        return H

Si carica la solita shape

In [3]:
n_qubits = 4
gates = { 
    1 : [ ["ID"]],
    2 : [ ["Sc","St"],["CZc","CZt"] ]
    ,3 : [ ["Sc","ID","St"] ]
    ,4 : [ ["Sc","Sc","St","St"] ] 
}

shape = CircuitShape(n_qubits,gates)
shape.print_steps(); 

0: ['ID', 'ID', 'ID', 'ID']
1: ['ID', 'ID', 'Sc', 'St']
2: ['ID', 'ID', 'CZc', 'CZt']
3: ['ID', 'Sc', 'St', 'ID']
4: ['ID', 'CZc', 'CZt', 'ID']
5: ['ID', 'Sc', 'ID', 'St']
6: ['Sc', 'St', 'ID', 'ID']
7: ['Sc', 'St', 'Sc', 'St']
8: ['Sc', 'St', 'CZc', 'CZt']
9: ['CZc', 'CZt', 'ID', 'ID']
10: ['CZc', 'CZt', 'Sc', 'St']
11: ['CZc', 'CZt', 'CZc', 'CZt']
12: ['Sc', 'ID', 'St', 'ID']
13: ['Sc', 'Sc', 'St', 'St']



E si prova, creato un CircuitOperations su tale shape, a spostare lo stato di un generico circuito single-step a ad un generico circuito single-step

In [4]:
c = CircuitOperations(shape)

steps1 = [
    ['CZc', 'CZt', 'ID', 'ID']
]
    
c1 = CircuitState(shape, steps=steps1)

print("Before:",c1.draw())
c.hop(4,c1)
print("After:",c1.draw())

Before:         
q_0: ─■─
      │ 
q_1: ─■─
        
q_2: ───
        
q_3: ───
        
After:         
q_0: ───
        
q_1: ─■─
      │ 
q_2: ─■─
        
q_3: ───
        


Si implementa poi un metodo che calcolate le equivalenze fino ad un certo ordine d calcola la hamiltoniana H che a evolvere secondo le equivalenze

In [5]:
H = c.get_H("../EQUIVALENZE/equivalenze v2",2,4)

Test della costruzione delle matrici H_eps della singola equivalenza

In [6]:
dir = "../EQUIVALENZE/equivalenze v2"
d=2
filename = f"{dir}/eq_d={d}.csv"
with open(filename, 'r', newline='') as fin:
    reader = csv.reader(fin, delimiter="\t")
    #skips the header
    row = None
    while(row != "equivalences:"):
        row = next(reader)[0]
    
    equivalences = [[int(id) for id in row] for row in reader]

equivalences[:4]

[[0, 0, 1, 1], [0, 0, 2, 2], [0, 0, 3, 3], [0, 0, 4, 4]]

In [112]:
eq1 = equivalences[0]
eq2 = equivalences[1]
H_eps1 = c.get_H_eps(eq1[:d],eq1[d:],4)
H_eps2 = c.get_H_eps(eq2[:d],eq2[d:],4)
H_eps = H_eps1 + H_eps2

c1 = CircuitState.from_steps_ids([1,1,0,0],shape)
print("C1:",c1.draw())
c2 =  CircuitOperations.multiply(H_eps,c1).normalize()
# print("Tutti zero:",all(c == 0 for c in c2._state))
print("C2 = H_eps C1:", c2.measure().draw())
print("Riapplicando a C2:", CircuitOperations.multiply(H_eps,c2).normalize().measure().draw())

C1:            
q_0: ──────
           
q_1: ──────
           
q_2: ─X──X─
      │  │ 
q_3: ─X──X─
           
C2 = H_eps C1:                  
q_0: ────────────
                 
q_1: ────────────
                 
q_2: ─X──X──X──X─
      │  │  │  │ 
q_3: ─X──X──X──X─
                 
Riapplicando a C2:            
q_0: ──────
           
q_1: ──────
           
q_2: ─X──X─
      │  │ 
q_3: ─X──X─
           
